# Stock Image Scraper 📸

In [28]:
# STEP 1:- Importing the required libraries
import time
import requests
import pandas as pd
from tqdm import tqdm
import chromedriver_binary
from bs4 import BeautifulSoup
from selenium import webdriver
from openpyxl import Workbook
import re

In [18]:
# STEP :-2:- Setting up the Selenium WebDriver
driver = webdriver.Chrome()
driver.get("https://stock-pictures.netlify.app/")
time.sleep(5)  # Wait for the page to load

The chromedriver version (147.0.7727.57) detected in PATH at c:\Users\Nbinary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\chromedriver_binary\chromedriver.exe might not be compatible with the detected chrome version (148.0.7778.181); currently, chromedriver 148.0.7778.178 is recommended for chrome 148.*, so it is advised to delete the driver in PATH and retry


In [33]:
# STEP 3:- Scrapping the url
soup = BeautifulSoup(driver.page_source, "html.parser")
image_elements = soup.select('img.source-img[src$=".jpg"]')
print(f"Found {len(image_elements)} images on the page.")

Found 146 images on the page.


In [34]:
# STEP 3:- Scraping the image details
# image url, name tags, likes and comments on each image
image_data = []
for sp in soup.find_all('div', class_='container'):
    img = sp.find('img')
    
    # 1. Safely check if img exists AND has a src attribute before proceeding
    if img and img.get('src') and 'gif' not in img.get('src'):
        link = img.get('src')
        
        # 2. Safely extract tags (Default to empty string if missing)
        tags_div = sp.find('div', class_='tags')
        tags_text = ""
        if tags_div:
            # Assuming the first 7 chars were something like "Tags: "
            # A safer way is replacing the specific word, or just matching words
            raw_tags = tags_div.text[7:].strip().split(' ')
            tags_text = ' '.join(list(set(raw_tags)))
        
        # 3. Safely extract likes and comments (Default to 0 if missing)
        likes, comments = 0, 0
        likes_comments_div = sp.find('div', class_='likes-comments')
        
        if likes_comments_div:
            spans = likes_comments_div.find_all('span')
            
            # re.search(r'\d+', text) finds the first sequence of numbers in a string
            if len(spans) > 0:
                likes_match = re.search(r'\d+', spans[0].text)
                likes = int(likes_match.group()) if likes_match else 0
                
            if len(spans) > 1:
                comments_match = re.search(r'\d+', spans[1].text)
                comments = int(comments_match.group()) if comments_match else 0
        
        image_data.append([link, tags_text, likes, comments])

In [35]:
# STEP 4: Make in to a dataframe
df = pd.DataFrame(image_data, columns=['Image URL', 'Tags', 'Likes', 'Comments'])

In [36]:
df

,Image URL,Tags,Likes,Comments
0,https://cdn.pixabay.com/photo/2022/03/06/05/30...,"Clouds, Blue Sky Atmosphere, Sky,",196,55
1,https://cdn.pixabay.com/photo/2022/04/07/11/45...,"Bird, Ornithology, Hummingbird",76,20
2,https://cdn.pixabay.com/photo/2022/02/28/15/28...,"Rainfall, Rainbow, Subtropical Sea,",282,106
3,https://cdn.pixabay.com/photo/2022/04/04/02/52...,"Cherry Road, Japan, Sakura Blossoms,",42,11
4,https://cdn.pixabay.com/photo/2022/04/09/18/06...,"Flower, Plant Marguerite, Cape",39,15
...,...,...,...,...
141,https://cdn.pixabay.com/photo/2022/03/28/20/47...,"view lake Composition,",81,81
142,https://cdn.pixabay.com/photo/2022/04/08/17/33...,"Bird, Blackbird, Songbird, Animal Fauna,",18,11
143,https://cdn.pixabay.com/photo/2022/04/10/10/04...,"Durian snow flat In to Japan, without inhabiti...",10,6
144,https://cdn.pixabay.com/photo/2021/11/09/07/29...,"Baby, Costume, Sleeping Newborn,",91,21


In [37]:
# STEP 5: Save the data to an Excel file
df.to_excel("stock_images_data.xlsx", index=False)